# **Embedding Test**

In [26]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path

## **Loading & Modifying the Model**

In [27]:
model = models.resnet50(weights="DEFAULT")
model.fc = torch.nn.Identity()
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

What you are looking at is the complete "X-ray" or blueprint of the ResNet50 factory assembly line.
But the most important part of this massive wall of text is at the very bottom. Look at the last two lines:
```

  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Identity()
```
Why this means you succeeded:

If you had just loaded a standard ResNet50, that very last line would have looked like this:
```
 (fc): Linear(in_features=2048, out_features=1000)
```
By seeing (fc): Identity(), you have absolute visual proof that you successfully "fired the Boss." You removed the 1,000-class ImageNet guessing layer and replaced it with an empty hallway (Identity).

## **Prepping the Image & Extracting the Embedding**

In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gpiosenka/butterfly-images40-species")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'butterfly-images40-species' dataset.
Path to dataset files: /kaggle/input/butterfly-images40-species


In [29]:
from pathlib import Path
img_path= Path(path) / 'train/ADONIS/001.jpg'
img = Image.open(img_path)

preprocess =transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
img_tensor = preprocess(img)
img_tensor = img_tensor.unsqueeze(0)

with torch.no_grad():
    embedding = model(img_tensor)
print(embedding.shape)

torch.Size([1, 2048])


# **Batch Embedding Extraction**

In [30]:
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = model.to(device)


train_dir = Path(path)/'train'
full_dataset = ImageFolder(root=train_dir, transform=preprocess)

batch_size = 32
dataloader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

all_embeddings = []
all_labels = []

for images, labels in dataloader:
  images = images.to(device)
  with torch.no_grad():
    embeddings = model(images)
    all_embeddings.append(embeddings.cpu().detach().numpy())
    all_labels.append(labels.numpy())

print('Loop finished!')


Using device: cuda
Loop finished!


In [31]:
final_embeddings = np.vstack(all_embeddings)
final_labels = np.concatenate(all_labels)

np.save('/content/embeddings.npy', final_embeddings)
np.save('/content/labels.npy', final_labels)

print("Saved embeddings.npy and labels.npy to /content!")
print(f"Final Embedding Matrix Shape: {final_embeddings.shape}")

Saved embeddings.npy and labels.npy to /content!
Final Embedding Matrix Shape: (12594, 2048)


## **Sanity Check On Embeddings**

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

embeddings = np.load('/content/embeddings.npy')
labels = np.load('/content/labels.npy')

In [33]:
#same species
bug_1 = embeddings[0].reshape(1, -1)
bug_2 = embeddings[1].reshape(1, -1)

print(cosine_similarity(bug_1, bug_2))


[[0.81406283]]


In [34]:
#diff species
bug_3 = embeddings[5000].reshape(1, -1)

print(cosine_similarity(bug_1, bug_3))


[[0.72518754]]


In [39]:
#Multiple pairs checking script
import random
unique_classes = np.unique(labels)

print('Same species pairs')
for i in range(15):
  test_sp= np.random.choice(unique_classes)
  test_idc= np.where(labels == test_sp)[0]

  bug_a = embeddings[random.choice(test_idc)].reshape(1, -1)
  bug_b = embeddings[random.choice(test_idc)].reshape(1, -1)

  cos_sim= cosine_similarity(bug_a, bug_b)[0][0]
  print(f'Class: {test_sp}, Cosine Similarity: {cos_sim}')

print('Different species pairs')
for i in range(15):
  test_sp_1= np.random.choice(unique_classes)
  test_sp_2= np.random.choice(unique_classes)

  test_idc_1= np.where(labels == test_sp_1)[0]
  test_idc_2= np.where(labels == test_sp_2)[0]

  bug_a = embeddings[random.choice(test_idc_1)].reshape(1, -1)
  bug_b = embeddings[random.choice(test_idc_2)].reshape(1, -1)

  cos_sim= cosine_similarity(bug_a, bug_b)[0][0]
  print(f'Class 1: {test_sp_1}, Class 2: {test_sp_2}, Cosine Similarity: {cos_sim}')


Same species pairs
Class: 49, Cosine Similarity: 0.8710916042327881
Class: 40, Cosine Similarity: 0.395230770111084
Class: 20, Cosine Similarity: 0.5892565250396729
Class: 32, Cosine Similarity: 0.7808694243431091
Class: 45, Cosine Similarity: 0.5292069911956787
Class: 12, Cosine Similarity: 0.685939610004425
Class: 87, Cosine Similarity: 0.5563892722129822
Class: 52, Cosine Similarity: 0.6139628887176514
Class: 2, Cosine Similarity: 0.6473721265792847
Class: 82, Cosine Similarity: 0.8440929055213928
Class: 82, Cosine Similarity: 0.7664178609848022
Class: 78, Cosine Similarity: 0.6967364549636841
Class: 42, Cosine Similarity: 0.5743898153305054
Class: 69, Cosine Similarity: 0.755885899066925
Class: 27, Cosine Similarity: 0.6060075759887695
Different species pairs
Class 1: 63, Class 2: 97, Cosine Similarity: 0.5159313082695007
Class 1: 41, Class 2: 51, Cosine Similarity: 0.34532445669174194
Class 1: 35, Class 2: 29, Cosine Similarity: 0.3145425021648407
Class 1: 45, Class 2: 95, Cosine 


**The Baseline Success**
The pipeline is functioning correctly. On average, intra-class pairs  maintained a high similarity score (0.60–0.85), while inter-class pairs dropped significantly (0.30–0.50). This distinct numerical gap proves the embeddings hold valid, separable visual meaning.

**The Anomalies**
The test also revealed critical edge cases that our future models will need to solve:
* **Low Intra-Class Similarity (e.g., Score 0.39 for the same species):** This occurs due to environmental variance. ResNet50 embeds the *entire* image. A specimen photographed on a sterile white background will score poorly against a specimen photographed in a dense, green forest, even if it is the same insect.
* **High Inter-Class Similarity (e.g., Score 0.80 for different species):** This points to visually identical species, likely due to biological mimicry or shared genus traits.